# 0. Overview

This notebook analyzes whether the ENSO-850 hPa wind relationship over the Maritime Continent changed between **Old (1981-2006)** and **New (2007-2025)** periods, compared against the **Full (1981-2025)** baseline.

Scientific objective:
- Separate mean-state change (climatology) from teleconnection change (correlation/regression vs Niño3.4) and event-conditioned responses (El Niño / La Niña) in the 850 hPa wind field.
- Plot wind speed as shaded fields and overlay wind vectors with quiver arrows.


# 1. Import Library


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.pyplot as plt


In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'


# 2. Open Data & Pre-Process


#### Open ERA5 850 hPa Wind


In [ ]:
era5_path = '../../external/ClimateData/era5-monthly/uv_850_global_1979-2025.nc'
era5_chunks = {'valid_time': 12}

ds_era5 = xr.open_dataset(era5_path, chunks=era5_chunks)[['u', 'v']]
if 'pressure_level' in ds_era5.coords or 'pressure_level' in ds_era5.dims:
    ds_era5 = ds_era5.sel(pressure_level=850)
ds_era5 = ds_era5.rename({'valid_time': 'time', 'latitude': 'lat', 'longitude': 'lon'})
ds_era5 = ds_era5.assign_coords(lon=(ds_era5.lon % 360)).sortby('lon')
# slice wilayah maritime continent samudra pasifik dan samudra hindia
ds_era5 = ds_era5.sel(
    time=slice('1980-12-01', '2025-02-01'),
    lat=slice(31, -31),
    lon=slice(60, 291),
)
# keep DJF only, with December assigned to the following DJF year
month_mask = ds_era5.time.dt.month.isin([12, 1, 2])
djf_year = xr.where(ds_era5.time.dt.month == 12, ds_era5.time.dt.year + 1, ds_era5.time.dt.year)
ds_era5 = ds_era5.sel(time=month_mask).assign_coords(djf_year=('time', djf_year.sel(time=month_mask).data))

u850 = ds_era5['u']
v850 = ds_era5['v']
wind_speed = np.hypot(u850, v850)


#### Open NINO34 index


In [ ]:
nino34_path = '../../external/ClimateData/index-monthly/nino34.anom.csv'
nino34_column = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'

df_nino34 = pd.read_csv(
    nino34_path,
    usecols=['Date', nino34_column],
    parse_dates=['Date'],
)
df_nino34 = df_nino34.set_index('Date').loc['1980-12-01':'2025-03-01']
df_nino34 = df_nino34[df_nino34.index.month.isin([12, 1, 2])].copy()
df_nino34['djf_year'] = df_nino34.index.year + (df_nino34.index.month == 12).astype('int8')
df_nino34['DJF'] = 'DJF' + df_nino34['djf_year'].astype(str)


#### Define Period


In [ ]:
full_years = np.arange(1981, 2026)
past_years = np.arange(1981, 2007)
recent_years = np.arange(2007, 2026)

wind_clim = wind_speed.sel(time=wind_speed.djf_year.isin(full_years))
wind_past = wind_clim.sel(time=wind_clim.djf_year.isin(past_years))
wind_recent = wind_clim.sel(time=wind_clim.djf_year.isin(recent_years))

u_clim = u850.sel(time=u850.djf_year.isin(full_years))
u_past = u_clim.sel(time=u_clim.djf_year.isin(past_years))
u_recent = u_clim.sel(time=u_clim.djf_year.isin(recent_years))

v_clim = v850.sel(time=v850.djf_year.isin(full_years))
v_past = v_clim.sel(time=v_clim.djf_year.isin(past_years))
v_recent = v_clim.sel(time=v_clim.djf_year.isin(recent_years))

nino34_clim = df_nino34[df_nino34['djf_year'].isin(full_years)].copy()
nino34_past = nino34_clim[nino34_clim['djf_year'].isin(past_years)].copy()
nino34_recent = nino34_clim[nino34_clim['djf_year'].isin(recent_years)].copy()

period_coord = xr.DataArray(
    np.where(wind_clim.djf_year <= 2006, 'past', 'recent'),
    coords={'time': wind_clim.time},
    dims='time',
    name='period',
)
wind_period = wind_clim.assign_coords(period=period_coord)
u_period = u_clim.assign_coords(period=period_coord)
v_period = v_clim.assign_coords(period=period_coord)


#### Define Area


In [ ]:
# Define the extent and ticks 
# ip = Indonesia-Pacific region
ip_extent = [60, 270, -30, 30]
ip_yticks = np.arange(-30, 31, 10)
ip_xticks = np.arange(60, 271, 30)


In [ ]:
# Define the extent and ticks 
# mc: maritime continent
mc_extent = [80, 160, -20, 20]  # [lon_min, lon_max, lat_min, lat_max]
mc_yticks = np.arange(-20, 21, 10)
mc_xticks = np.arange(90, 161, 20)


In [ ]:
# Compute the shared climatology means once and reuse the period grouping
full_plot, period_means, full_u_plot, period_u_means, full_v_plot, period_v_means = dask.compute(
    wind_clim.mean('time'),
    wind_period.groupby('period').mean('time'),
    u_clim.mean('time'),
    u_period.groupby('period').mean('time'),
    v_clim.mean('time'),
    v_period.groupby('period').mean('time'),
)

# Transpose after loading (faster on in-memory numpy arrays)
full_plot = full_plot.transpose('lat', 'lon')
past_plot = period_means.sel(period='past').transpose('lat', 'lon')
recent_plot = period_means.sel(period='recent').transpose('lat', 'lon')
diff_plot = recent_plot - past_plot

full_u_plot = full_u_plot.transpose('lat', 'lon')
past_u_plot = period_u_means.sel(period='past').transpose('lat', 'lon')
recent_u_plot = period_u_means.sel(period='recent').transpose('lat', 'lon')
diff_u_plot = recent_u_plot - past_u_plot

full_v_plot = full_v_plot.transpose('lat', 'lon')
past_v_plot = period_v_means.sel(period='past').transpose('lat', 'lon')
recent_v_plot = period_v_means.sel(period='recent').transpose('lat', 'lon')
diff_v_plot = recent_v_plot - past_v_plot


In [ ]:


def _add_quiver(ax, u, v, step=16, scale=70, width=0.002, color='k', key_u=10, key_y=1.1, key_label=None, key_color='k'):
    u_plot = u.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    v_plot = v.isel(lat=slice(None, None, step), lon=slice(None, None, step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=width,
        color=color,
        zorder=4,
    )
    if key_label is None:
        key_label = f'{key_u} m/s'
    ax.quiverkey(q, X=0.88, Y=key_y, U=key_u, label=key_label, labelpos='E', coordinates='axes', color=key_color)
    return q


def add_quiver_clim(ax, u, v, step=16, scale=70, width=0.002, color='k', key_y=1.1, key_u=10, key_label=None):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, key_u=key_u, key_y=key_y, key_label=key_label)


def add_quiver_anom(ax, u, v, step=16, scale=70, width=0.002, key_u=2, color='gray', key_color='black', key_label=None, key_y=1.1):
    return _add_quiver(ax, u, v, step=step, scale=scale, width=width, color=color, 
                       key_u=key_u, key_y=key_y, key_label=key_label, key_color=key_color)


# 4. Plot Composite


### A. Analisis El Nino


In [ ]:
# define el nino years based on DJF-mean nino34 index threshold
elnino_threshold = 0.5
nino34_clim_djf = nino34_clim.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_past_djf = nino34_past.groupby('djf_year')[nino34_column].mean().reset_index()
nino34_recent_djf = nino34_recent.groupby('djf_year')[nino34_column].mean().reset_index()

nino34_clim_elnino = nino34_clim_djf[nino34_clim_djf[nino34_column] >= elnino_threshold]
nino34_past_elnino = nino34_past_djf[nino34_past_djf[nino34_column] >= elnino_threshold]
nino34_recent_elnino = nino34_recent_djf[nino34_recent_djf[nino34_column] >= elnino_threshold]

elnino_years_clim = sorted(nino34_clim_elnino['djf_year'].tolist())
elnino_years_past = sorted(nino34_past_elnino['djf_year'].tolist())
elnino_years_recent = sorted(nino34_recent_elnino['djf_year'].tolist())
print('El Nino DJF years (1981-2025):', elnino_years_clim)
print('El Nino DJF years (1981-2006):', elnino_years_past)
print('El Nino DJF years (2007-2025):', elnino_years_recent)

u_clim_elnino = u_clim.sel(time=u_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
u_past_elnino = u_past.sel(time=u_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
u_recent_elnino = u_recent.sel(time=u_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

v_clim_elnino = v_clim.sel(time=v_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
v_past_elnino = v_past.sel(time=v_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
v_recent_elnino = v_recent.sel(time=v_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

wind_clim_elnino = wind_clim.sel(time=wind_clim.djf_year.isin(elnino_years_clim)).mean('time').transpose('lat', 'lon')
wind_past_elnino = wind_past.sel(time=wind_past.djf_year.isin(elnino_years_past)).mean('time').transpose('lat', 'lon')
wind_recent_elnino = wind_recent.sel(time=wind_recent.djf_year.isin(elnino_years_recent)).mean('time').transpose('lat', 'lon')

u_clim_elnino_anom = (u_clim_elnino - full_u_plot).reset_coords(drop=True)
u_past_elnino_anom = (u_past_elnino - past_u_plot).reset_coords(drop=True)
u_recent_elnino_anom = (u_recent_elnino - recent_u_plot).reset_coords(drop=True)
diff_u_elnino_anom = (u_recent_elnino_anom - u_past_elnino_anom).reset_coords(drop=True)

v_clim_elnino_anom = (v_clim_elnino - full_v_plot).reset_coords(drop=True)
v_past_elnino_anom = (v_past_elnino - past_v_plot).reset_coords(drop=True)
v_recent_elnino_anom = (v_recent_elnino - recent_v_plot).reset_coords(drop=True)
diff_v_elnino_anom = (v_recent_elnino_anom - v_past_elnino_anom).reset_coords(drop=True)

wind_clim_elnino_anom = (wind_clim_elnino - full_plot).reset_coords(drop=True)
wind_past_elnino_anom = (wind_past_elnino - past_plot).reset_coords(drop=True)
wind_recent_elnino_anom = (wind_recent_elnino - recent_plot).reset_coords(drop=True)
diff_elnino_anom = (wind_recent_elnino_anom - wind_past_elnino_anom).reset_coords(drop=True)


#### plot komposit indo pasifik


In [ ]:
# data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (wind_clim_elnino_anom, u_clim_elnino_anom, v_clim_elnino_anom, 'Komposit Angin 850 hPa El Nino DJF 1981-2025', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_past_elnino_anom, u_past_elnino_anom, v_past_elnino_anom, 'Komposit Angin 850 hPa El Nino DJF 1981-2006', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_recent_elnino_anom, u_recent_elnino_anom, v_recent_elnino_anom, 'Komposit Angin 850 hPa El Nino DJF 2007-2025', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (diff_elnino_anom, diff_u_elnino_anom, diff_v_elnino_anom, 'Selisih Komposit Angin 850 hPa El Nino DJF P2 - P1', 'Spectral', 'Selisih anomali kecepatan angin (m/s)', np.arange(-3, 3.1, 0.5), np.arange(-3, 3.1, 1), 'both'),
]

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=20, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot komposit maritime continent


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=10, scale=280, width=0.002, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


### B. Analisis La Nina


In [ ]:
# define la nina years based on DJF-mean nino34 index threshold
lanina_threshold = -0.5

nino34_clim_lanina = nino34_clim_djf[nino34_clim_djf[nino34_column] <= lanina_threshold]
nino34_past_lanina = nino34_past_djf[nino34_past_djf[nino34_column] <= lanina_threshold]
nino34_recent_lanina = nino34_recent_djf[nino34_recent_djf[nino34_column] <= lanina_threshold]

lanina_years_clim = sorted(nino34_clim_lanina['djf_year'].tolist())
lanina_years_past = sorted(nino34_past_lanina['djf_year'].tolist())
lanina_years_recent = sorted(nino34_recent_lanina['djf_year'].tolist())
print('La Nina DJF years (1981-2025):', lanina_years_clim)
print('La Nina DJF years (1981-2006):', lanina_years_past)
print('La Nina DJF years (2007-2025):', lanina_years_recent)

u_clim_lanina = u_clim.sel(time=u_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
u_past_lanina = u_past.sel(time=u_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
u_recent_lanina = u_recent.sel(time=u_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

v_clim_lanina = v_clim.sel(time=v_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
v_past_lanina = v_past.sel(time=v_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
v_recent_lanina = v_recent.sel(time=v_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

wind_clim_lanina = wind_clim.sel(time=wind_clim.djf_year.isin(lanina_years_clim)).mean('time').transpose('lat', 'lon')
wind_past_lanina = wind_past.sel(time=wind_past.djf_year.isin(lanina_years_past)).mean('time').transpose('lat', 'lon')
wind_recent_lanina = wind_recent.sel(time=wind_recent.djf_year.isin(lanina_years_recent)).mean('time').transpose('lat', 'lon')

u_clim_lanina_anom = (u_clim_lanina - full_u_plot).reset_coords(drop=True)
u_past_lanina_anom = (u_past_lanina - past_u_plot).reset_coords(drop=True)
u_recent_lanina_anom = (u_recent_lanina - recent_u_plot).reset_coords(drop=True)
diff_u_lanina_anom = (u_recent_lanina_anom - u_past_lanina_anom).reset_coords(drop=True)

v_clim_lanina_anom = (v_clim_lanina - full_v_plot).reset_coords(drop=True)
v_past_lanina_anom = (v_past_lanina - past_v_plot).reset_coords(drop=True)
v_recent_lanina_anom = (v_recent_lanina - recent_v_plot).reset_coords(drop=True)
diff_v_lanina_anom = (v_recent_lanina_anom - v_past_lanina_anom).reset_coords(drop=True)

wind_clim_lanina_anom = (wind_clim_lanina - full_plot).reset_coords(drop=True)
wind_past_lanina_anom = (wind_past_lanina - past_plot).reset_coords(drop=True)
wind_recent_lanina_anom = (wind_recent_lanina - recent_plot).reset_coords(drop=True)
diff_lanina_anom = (wind_recent_lanina_anom - wind_past_lanina_anom).reset_coords(drop=True)


#### plot komposit indo pasifik


In [ ]:
# data, title, cmap, levels, ticks, cbar_label, extend
plots = [
    (wind_clim_lanina_anom, u_clim_lanina_anom, v_clim_lanina_anom, 'Komposit Angin 850 hPa La Nina DJF 1981-2025', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_past_lanina_anom, u_past_lanina_anom, v_past_lanina_anom, 'Komposit Angin 850 hPa La Nina DJF 1981-2006', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (wind_recent_lanina_anom, u_recent_lanina_anom, v_recent_lanina_anom, 'Komposit Angin 850 hPa La Nina DJF 2007-2025', 'Spectral', 'Anomali kecepatan angin (m/s)', np.arange(-5, 5.1, 0.5), np.arange(-5, 5.1, 1), 'both'),
    (diff_lanina_anom, diff_u_lanina_anom, diff_v_lanina_anom, 'Selisih Komposit Angin 850 hPa La Nina DJF P2 - P1', 'Spectral', 'Selisih anomali kecepatan angin (m/s)', np.arange(-3, 3.1, 0.5), np.arange(-3, 3.1, 1), 'both'),
]

for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(12, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=20, scale=50, width=0.0017, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(ip_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(ip_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(ip_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()


#### plot komposit maritime continent


In [ ]:
for data, u_data, v_data, title, cmap, cbar_label, levels, ticks, extend in plots:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        levels=levels,
        extend=extend,
        add_colorbar=False,
        infer_intervals=True,
    )

    add_quiver_clim(ax, u_data, v_data, step=10, scale=280, width=0.002, key_u=1, key_label='1 m/s', key_y=1.05)

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=ticks, spacing='proportional', extend=extend)
    cbar.set_label(cbar_label, fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    plt.show()
